# Silver: phenotype findings out of the clinical notes

Reads the free-text notes and produces coded findings from them, each one carrying the
note it came from and the character span it was matched on.

| | |
| --- | --- |
| **Reads** | `bronze_clinical_notes` |
| **Writes** | `silver_extracted_findings`, `silver_extraction_quality` |

## Why the extractor is deterministic

A note says *low tone*, never *HP:0001252*. Something has to bridge that, and the
obvious reach is a language model.

This uses dictionary matching with assertion detection instead, because the whole
argument of this pipeline is that **the criteria decide who surfaces, and the criteria
are inspectable**. Put a model in the extraction path and it is a model that decides
which findings a patient has -- which is the same thing one step earlier, and no longer
reproducible, argued with, or measurable for bias.

A production build would use clinical NLP here, and should. The contract is what
matters and it does not change: every extracted finding names the note and the span it
came from, so a clinician can read the sentence and disagree with it.

## Assertion is the hard part, not matching

Matching *scoliosis* is trivial. The three sentences below all contain it and only one
is a finding about this patient:

* `Scoliosis noted, unchanged since the last review.` -> **present**
* `No evidence of scoliosis.` -> **negated**
* `An older sibling was investigated for scoliosis.` -> **family history**

Only `present` reaches Gold. A dictionary match without assertion detection would
report a spinal curvature for a patient whose notes say they do not have one, and
inflate the very body-system count the strongest criterion reads.

In [ ]:
# Characters either side of a match to search for a negation or family-history cue.
# Long enough to catch "no evidence of X" before and "X was ruled out" after, short
# enough not to cross into the neighbouring sentence.
CUE_WINDOW = 42
CUE_WINDOW_AFTER = 34
PIPELINE_RUN_ID = ""

In [ ]:
import re

import notebookutils
from pyspark.sql import functions as F
from pyspark.sql.types import (ArrayType, IntegerType, StringType, StructField,
                               StructType)

RUN_ID = PIPELINE_RUN_ID or "local"
BRONZE = "bronze_lakehouse"

_WS = notebookutils.runtime.context["currentWorkspaceId"]
_ONELAKE = notebookutils.conf.get("trident.onelake.endpoint").replace("https://", "")
_LAKEHOUSE_ID = {}


def lake_table(lakehouse, table):
    """Read from a lakehouse other than the attached default -- by OneLake path, which
    needs the lakehouse *id*; passing the name returns an opaque ABFS 400."""
    if lakehouse not in _LAKEHOUSE_ID:
        _LAKEHOUSE_ID[lakehouse] = notebookutils.lakehouse.get(
            lakehouse, workspaceId=_WS).id
    return spark.read.format("delta").load(
        f"abfss://{_WS}@{_ONELAKE}/{_LAKEHOUSE_ID[lakehouse]}/Tables/{table}")


notes = lake_table(BRONZE, "bronze_clinical_notes")
print(f"notes  {notes.count():,}")

In [ ]:
# ------------------------------------------------------------------- lexicon
# Surface forms a clinician actually writes, mapped to the HPO term they mean. Ordered
# longest-first at match time so "sensorineural hearing loss" wins over "hearing loss"
# and the span points at the whole phrase.
LEXICON = {
    "HP:0001263": ["global developmental delay", "developmental delay",
                   "delayed milestones", "delay across all developmental domains"],
    "HP:0001249": ["intellectual disability", "cognitive impairment",
                   "significant learning difficulties", "learning difficulties"],
    "HP:0000750": ["delayed speech and language", "speech delay",
                   "very limited expressive language", "limited expressive language"],
    "HP:0002376": ["developmental regression", "loss of previously acquired skills",
                   "regression of milestones"],
    "HP:0001252": ["reduced muscle tone", "hypotonia", "low tone", "a floppy infant",
                   "floppy infant"],
    "HP:0001250": ["seizure activity", "convulsive episodes", "seizures", "seizure"],
    "HP:0004322": ["height below the third centile", "growth restriction",
                   "short stature"],
    "HP:0001518": ["small for gestational age", "low birth weight for dates"],
    "HP:0011968": ["ongoing difficulty feeding", "feeding difficulties", "poor feeding"],
    "HP:0000252": ["OFC below the second centile", "a small head circumference",
                   "small head circumference", "microcephaly"],
    "HP:0000175": ["a palatal cleft", "palatal cleft", "cleft palate"],
    "HP:0001999": ["an unusual facial appearance", "unusual facial appearance",
                   "dysmorphic features", "facial dysmorphism"],
    "HP:0001627": ["a structural cardiac anomaly", "structural cardiac anomaly",
                   "abnormal heart morphology", "a congenital heart defect",
                   "congenital heart defect"],
    "HP:0000365": ["sensorineural hearing loss", "hearing impairment", "hearing loss"],
    "HP:0000505": ["reduced visual acuity", "visual impairment", "poor vision"],
    "HP:0002650": ["a scoliotic curve", "scoliotic curve", "curvature of the spine",
                   "scoliosis"],
}

NEGATION_CUES = ["no evidence of", "negative for", "ruled out", "denies", "deny",
                 "no sign of", "without", "not demonstrate", "no "]
# Cues that follow the term rather than precede it. "Scoliosis was considered and
# ruled out" reads as a finding to any backwards-only window, and asserting a
# spinal curvature for a patient whose note rules one out is the worst direction
# for this error to run in.
NEGATION_CUES_AFTER = ["was considered and ruled out", "was ruled out",
                       "ruled out", "was excluded", "not present",
                       "was not demonstrated"]
FAMILY_CUES = ["family history of", "mother has", "mother had", "father reports",
               "father has", "sibling was", "sibling has", "maternal cousin",
               "in childhood", "brother", "sister", "cousin"]

# Case-folded, because matching runs against lowered text -- a surface form carrying
# capitals ("OFC below the second centile") would otherwise never fire at all.
# Longest first, so the span covers the fullest phrase that matched.
SURFACE = sorted(((s.lower(), term) for term, forms in LEXICON.items()
                  for s in forms), key=lambda pair: -len(pair[0]))
print(f"lexicon  {len(SURFACE)} surface forms over {len(LEXICON)} terms")

In [ ]:
# ----------------------------------------------------------------- extraction
finding_schema = ArrayType(StructType([
    StructField("hpo_id", StringType()),
    StructField("matched_text", StringType()),
    StructField("char_start", IntegerType()),
    StructField("char_end", IntegerType()),
    StructField("assertion", StringType()),
    StructField("cue", StringType()),
]))


def assert_type(text, start, end):
    """present / negated / family_history, from the words around a match.

    Family history is checked first: "no family history of X" is still a statement
    about the family, and must not be counted as a finding about this patient either
    way. Both non-present classes are kept rather than dropped, so the note can show
    what it decided and why.
    """
    # Clip both windows at the sentence boundary. Without this, "poor vision.
    # Considered and ruled out: delayed milestones" negates the poor vision -- the cue
    # belongs to the next sentence and says nothing about this finding.
    before = text[max(0, start - CUE_WINDOW):start].lower().rsplit(".", 1)[-1]
    after = text[end:end + CUE_WINDOW_AFTER].lower().split(".", 1)[0]
    for cue in FAMILY_CUES:
        if cue in before:
            return "family_history", cue
    for cue in NEGATION_CUES:
        if cue in before:
            return "negated", cue
    for cue in NEGATION_CUES_AFTER:
        if cue in after:
            return "negated", cue
    return "present", None


def extract(text):
    if not text:
        return []
    lowered = text.lower()
    out, claimed = [], []
    for surface, term in SURFACE:
        start = 0
        while True:
            hit = lowered.find(surface, start)
            if hit < 0:
                break
            end = hit + len(surface)
            # A longer form already covered this span; do not also emit the short one.
            if any(hit < c_end and end > c_start for c_start, c_end in claimed):
                start = end
                continue
            claimed.append((hit, end))
            assertion, cue = assert_type(text, hit, end)
            out.append({"hpo_id": term, "matched_text": text[hit:end],
                        "char_start": hit, "char_end": end,
                        "assertion": assertion, "cue": cue})
            start = end
    return sorted(out, key=lambda f: f["char_start"])


extract_udf = F.udf(extract, finding_schema)

extracted = (notes
             .withColumn("finding", F.explode(extract_udf(F.col("note_text"))))
             .select(
                 F.col("note_id"), F.col("patient_id"), F.col("encounter_id"),
                 F.col("note_date_raw"), F.col("author_specialty_raw"),
                 F.col("finding.hpo_id").alias("hpo_id"),
                 F.col("finding.matched_text").alias("matched_text"),
                 F.col("finding.char_start").alias("char_start"),
                 F.col("finding.char_end").alias("char_end"),
                 F.col("finding.assertion").alias("assertion"),
                 F.col("finding.cue").alias("assertion_cue"),
                 F.lit(RUN_ID).alias("run_id"))
             .withColumn("extraction_method", F.lit("lexicon+assertion/v1")))

extracted.write.mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable("silver_extracted_findings")
print(f"silver_extracted_findings  {extracted.count():,} rows")
extracted.groupBy("assertion").count().orderBy(F.desc("count")).show(truncate=False)

In [ ]:
# ------------------------------------------------------------------- quality
# The synthetic cohort carries an answer key, so extraction can be measured rather
# than asserted. A real deployment cannot do this -- which is the argument for having
# built a synthetic cohort in the first place.
truth = lake_table(BRONZE, "_bronze_note_truth")
got = spark.table("silver_extracted_findings")

key = ["note_id", "hpo_id", "assertion"]
tp = truth.select(*key).distinct().join(got.select(*key).distinct(), key).count()
in_truth = truth.select(*key).distinct().count()
in_got = got.select(*key).distinct().count()

precision = tp / in_got if in_got else 0.0
recall = tp / in_truth if in_truth else 0.0

# The number that actually matters: findings that exist only in prose, and how many of
# them extraction recovered. Everything else the coded feed already had.
only = truth.filter(F.col("origin") == "note_only").select("note_id", "hpo_id").distinct()
only_found = only.join(
    got.filter(F.col("assertion") == "present").select("note_id", "hpo_id").distinct(),
    ["note_id", "hpo_id"]).count()
only_total = only.count()

rows = [
    ("precision", "overall", float(round(precision, 4)), "ratio",
     "Extracted findings that match the answer key on note, term and assertion."),
    ("recall", "overall", float(round(recall, 4)), "ratio",
     "Answer-key findings the extractor recovered."),
    ("note_only_recall", "note_only", float(round(only_found / only_total, 4)), "ratio",
     "Share of findings present ONLY in prose that extraction recovered. These are "
     "invisible to the coded pipeline, so this is the number the notes stage exists "
     "to move."),
    ("note_only_total", "note_only", float(only_total), "findings",
     "Findings written in a note and never coded anywhere."),
    ("note_only_recovered", "note_only", float(only_found), "findings",
     "Of those, the ones extraction found and asserted as present."),
]
quality_schema = StructType([
    StructField("metric", StringType()), StructField("dimension", StringType()),
    StructField("value", StringType()), StructField("unit", StringType()),
    StructField("note", StringType()),
])
spark.createDataFrame(
    [(m, d, str(v), u, n) for m, d, v, u, n in rows], quality_schema) \
    .withColumn("value", F.col("value").cast("double")) \
    .withColumn("run_id", F.lit(RUN_ID)) \
    .write.mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable("silver_extraction_quality")

print(f"precision          {precision:.1%}")
print(f"recall             {recall:.1%}")
print(f"note-only recall   {only_found}/{only_total} = {only_found / only_total:.1%}")

if precision < 0.9 or recall < 0.85:
    raise ValueError(
        f"extraction quality below the gate (precision {precision:.1%}, "
        f"recall {recall:.1%}). Gold reads these findings as evidence, so shipping "
        f"them unmeasured would put unverified findings in a clinical brief.")
print("silver note extraction complete")